# Example 03: Two-terminal square lattice with a top non-Hermitian bath

This notebook extends the two-terminal square-lattice setup by adding an **extra bath** on the **upper edge** of the device through the `:nonhermitian_retarded` interface in `ExperimentalBlockRHSParams`.

In [ ]:
using Pkg
Pkg.activate(joinpath(dirname(dirname(@__DIR__))))

using TDNEGF
using DifferentialEquations
using LinearAlgebra
using PyPlot

println("Number of Julia threads: ", Threads.nthreads())
println("Number of BLAS threads: ", BLAS.get_num_threads())

## Geometry and bath setup

We use a finite square-lattice device with:
- `Nx = 4`, `Ny = 2`
- spinful local Hilbert space (`Nσ = 2`, `N_orb = 1`)
- standard left/right leads (same construction as the two-terminal example)
- one extra bath coupled only to the **top edge** (`y = Ny`)

The local retarded self-energy used for the extra bath is

\[
\Sigma^R_{\mathrm{loc}} = -i\,(\gamma_0 I + \gamma_y\,\sigma_y),
\]

embedded on the coupled top-edge indices.

In [ ]:
# Helpers following the same style as the two-terminal example.

local_index(x::Int, y::Int, Ny::Int) = (x - 1) * Ny + y

function local_subrange(site::Int, N_loc::Int)
    a = (site - 1) * N_loc + 1
    b = site * N_loc
    return a:b
end

function top_edge_coupled_indices(;Nx::Int, Ny::Int, N_loc::Int)
    idx = Int[]
    top_sites = [local_index(x, Ny, Ny) for x in 1:Nx]
    for s in top_sites
        append!(idx, collect(local_subrange(s, N_loc)))
    end
    return top_sites, idx
end

function build_top_bath_ΣR(;Nx::Int, Ny::Int, N_loc::Int, gamma0::Float64, gammay::Float64)
    @assert N_loc == 2 "This pedagogical example assumes N_loc=2 (spin-1/2, one orbital)."
    σy = ComplexF64[0.0 -1im; 1im 0.0]
    Σ_site = -1im .* (gamma0 .* Matrix{ComplexF64}(I, N_loc, N_loc) .+ gammay .* σy)
    ΣR_loc = kron(Matrix{ComplexF64}(I, Nx, Nx), Σ_site)
    return Σ_site, ΣR_loc
end

## Parameter choices

We run two short simulations:
1. `gamma0 > 0`, `gammay = 0` (spin-independent damping on the top edge)
2. same `gamma0`, but `gammay ≠ 0` to activate the spin-selective component

In [ ]:
function init_two_terminal_with_top_bath(;Nx::Int=4, Ny::Int=2, Nσ::Int=2, N_orb::Int=1,
                                         γ::Float64=1.0, γso=0.5 + 0.0im,
                                         N_λ1::Int=49, N_λ2::Int=20, β::Float64=33.0,
                                         gamma0::Float64=0.20, gammay::Float64=0.0)
    Rλ, zλ = load_poles_square(N_λ1, N_λ2)

    p_model = ModelParamsTDNEGF(Nx=Nx, Ny=Ny, Nσ=Nσ, N_orb=N_orb,
                                Nα=2, N_λ1=N_λ1, N_λ2=N_λ2)

    H_ab = build_H_ab(; Nx=p_model.Nx, Ny=p_model.Ny, Nσ=p_model.Nσ,
                      N_orb=p_model.N_orb, γ=γ, γso=γso)

    Σᴸ_nλ = build_Σᴸ_nλ(Rλ, zλ, p_model.Ny, p_model.Nσ, p_model.N_orb,
                       p_model.N_λ1, p_model.N_λ2; β=β, γ=1.0)
    Σᴳ_nλ = build_Σᴳ_nλ(Rλ, zλ, p_model.Ny, p_model.Nσ, p_model.N_orb,
                       p_model.N_λ1, p_model.N_λ2; β=β, γ=1.0)
    χ_nλ = build_χ_nλ(zλ, p_model.Ny, p_model.Nσ, p_model.N_orb,
                      p_model.N_λ1, p_model.N_λ2; β=β, γ=1.0)

    ξ_anR = build_ξ_an(p_model.Nx, p_model.Ny, p_model.Nσ, p_model.N_orb;
                       xcol=p_model.Nx, y_coup=1:p_model.Ny)
    ξ_anL = build_ξ_an(p_model.Nx, p_model.Ny, p_model.Nσ, p_model.N_orb;
                       xcol=1, y_coup=1:p_model.Ny)

    left_block = SelfEnergyBlock(:left, p_model.Nc, p_model.N_λ1, p_model.N_λ2,
                                 Σᴸ_nλ, Σᴳ_nλ, χ_nλ, ξ_anL)
    right_block = SelfEnergyBlock(:right, p_model.Nc, p_model.N_λ1, p_model.N_λ2,
                                  Σᴸ_nλ, Σᴳ_nλ, χ_nλ, ξ_anR)

    blocks = [left_block, right_block]
    Δ_blocks = ComplexF64[0.5 + 0.0im, -0.5 + 0.0im]

    top_sites, idx_top = top_edge_coupled_indices(;Nx=p_model.Nx, Ny=p_model.Ny, N_loc=p_model.N_loc)
    Σ_site, ΣR_loc = build_top_bath_ΣR(;Nx=p_model.Nx, Ny=p_model.Ny, N_loc=p_model.N_loc,
                                       gamma0=gamma0, gammay=gammay)

    top_bath = TDNEGF.ExtraBathSpec(:nonhermitian_retarded, :top_edge_nonhermitian, idx_top, ΣR_loc, true)

    p_model.H_ab .= H_ab
    p_model.H0_ab .= H_ab

    p_blocks = ExperimentalBlockRHSParams(p_model.H_ab, blocks, Δ_blocks, p_model, [top_bath])

    u0 = zeros(ComplexF64, p_blocks.dims_ρ_ab[1]^2 + p_blocks.aux_layout.total_size)

    return (p_model=p_model, p_blocks=p_blocks, u0=u0,
            top_sites=top_sites, idx_top=idx_top, Σ_site=Σ_site, ΣR_loc=ΣR_loc)
end

function run_short_evolution(;gamma0::Float64=0.20, gammay::Float64=0.0,
                             tspan=(0.0, 25.0), reltol=1e-6, abstol=1e-8)
    setup = init_two_terminal_with_top_bath(;gamma0=gamma0, gammay=gammay)
    p_model, p_blocks, u0 = setup.p_model, setup.p_blocks, setup.u0

    prob = ODEProblem(eom_tdnegf_blocks!, u0, tspan, p_blocks)
    sol = solve(prob, Vern7(), adaptive=true, dense=false, reltol=reltol, abstol=abstol)

    obs = ObservablesTDNEGF(p_model; N_tmax=length(sol.t), N_leads=length(p_blocks.blocks))
    obs.t = sol.t

    for (it, ut) in enumerate(sol.u)
        obs.idx = it
        dv = pointer_blocks(ut, p_blocks.dims_ρ_ab, p_blocks.aux_layout)
        obs_n_i!(dv, p_model, obs)
        obs_Ixα!(dv, p_blocks, obs)
    end

    return setup, sol, obs
end

In [ ]:
# First run: spin-independent top-edge damping (gammay = 0)
setup0, sol0, obs0 = run_short_evolution(;gamma0=0.20, gammay=0.0)

println("Top-edge site indices      = ", setup0.top_sites)
println("Top-edge coupled DOF idx   = ", setup0.idx_top)
println("length(idx_top)            = ", length(setup0.idx_top))
println("size(ΣR_loc)               = ", size(setup0.ΣR_loc))
println("Consistency check          = ", size(setup0.ΣR_loc, 1) == length(setup0.idx_top))

In [ ]:
# Second run: activate spin-selective component (gammay ≠ 0)
setupy, soly, obsy = run_short_evolution(;gamma0=0.20, gammay=0.10)

println("Activated bath with gammay = 0.10")
println("Local Σ_site used for each top-edge site:")
display(setupy.Σ_site)

## Simple observable

As a compact diagnostic, we compare the average density on the **top edge** as a function of time:

\[
\bar n_{\mathrm{top}}(t) = \frac{1}{N_x}\sum_{x=1}^{N_x} n_{(x, y=Ny)}(t).
\]

This highlights the effect of adding a spin-selective term (`gammay ≠ 0`) on top of isotropic damping.

In [ ]:
function top_edge_density_trace(obs, top_sites)
    n_top = zeros(length(obs.t))
    for s in top_sites
        n_top .+= vec(obs.n_i[s, :])
    end
    return n_top ./ length(top_sites)
end

n_top_0 = top_edge_density_trace(obs0, setup0.top_sites)
n_top_y = top_edge_density_trace(obsy, setupy.top_sites)

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.plot(obs0.t, n_top_0, label=L"$\gamma_y = 0$")
ax.plot(obsy.t, n_top_y, "--", label=L"$\gamma_y = 0.10$")
ax.set_xlabel(L"$t$")
ax.set_ylabel(L"$\bar n_{\mathrm{top}}(t)$")
ax.set_title("Top-edge density with non-Hermitian extra bath")
ax.legend(frameon=false)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Optional: compare right-lead charge current as an additional check.
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.plot(obs0.t, obs0.Iα[1, :] .* pi, label=L"$I_R,\ \gamma_y=0$")
ax.plot(obsy.t, obsy.Iα[1, :] .* pi, "--", label=L"$I_R,\ \gamma_y=0.10$")
ax.set_xlabel(L"$t$")
ax.set_ylabel(L"$I_R\ (2e/h)$")
ax.legend(frameon=false)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()